In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

# ===============================
# 1. Загрузка данных
# ===============================
train = pd.read_csv('data.csv')
test = pd.read_csv('test.csv')

sample_sub = pd.read_csv("solution_example_full.csv")

# ===============================
# 2. Удаляем NA из train
# ===============================
train = train.dropna().reset_index(drop=True)

def clean_data(df):

    df = df.copy()
    df = df.drop_duplicates()

    for col in df.select_dtypes(include=[np.number]):
        df[col] = df[col].fillna(df[col].median())

    for col in df.select_dtypes(include=["object", "string"]):
        df[col] = df[col].fillna("unknown")

    return df


train = clean_data(train)
test = clean_data(test)

# ===============================
# 3. Отделяем таргет
# ===============================
TARGET = 'price'
X = train.drop(columns=[TARGET])
y = train[TARGET]

agg = train.groupby('district_name')['price'].agg(['median','count'])

# ===============================
# 4. Удаляем выбросы по таргету (±3σ)
# ===============================
y_mean = y.mean()
y_std = y.std()

lower = y_mean - 3 * y_std
upper = y_mean + 1 * y_std
q_low = y.quantile(0.1)
q_high = y.quantile(0.95)

mask = (y >= q_low) & (y <= q_high)

# X = X[mask].reset_index(drop=True)
# y = y[mask].reset_index(drop=True)

print(f'После фильтрации выбросов осталось объектов: {len(y)}')

# ===============================
# 5. Feature Engineering
# ===============================
def add_features(df):
    df = df.copy()

    df['total_area'] = df["kitchen_area"]+df["bath_area"]+df["other_area"]
    df["total_area_s"]=df["total_area"]**2

    df['area_per_room'] = df['total_area'] / (df['rooms_count'] + 1)
    df["area_per_room_s"] = df["area_per_room"]**2
    df['bath_per_room'] = df['bath_count'] / (df['rooms_count'] + 1)

    df['is_top_floor'] = (df['floor'] == df['floor_max']).astype(int)
    df['is_first_floor'] = (df['floor'] == 1).astype(int)

    df['kitchen_ratio'] = df['kitchen_area'] / (df['total_area'] + 1)
    df['bath_ratio'] = df['bath_area'] / (df['total_area'] + 1)

    # СЕГМЕНТНЫЕ ФИЧИ
    df['is_studio'] = (df['rooms_count'] == 0).astype(int)
    df['is_small'] = (df['total_area'] < 50).astype(int)
    df['is_large'] = (df['total_area'] > 80).astype(int)


    # df['district_median_price'] = df['district_name'].map(agg['median'])

    return df

X = add_features(X)
test = add_features(test)

# ===============================
# 6. Log-трансформация таргета
# ===============================
y_log = np.log1p(y)

# ===============================
# 7. Категориальные признаки
# ===============================
cat_features = [
    'district_name',
    'extra_area_type_name',
    'gas',
    'hot_water',
    'central_heating'
]

# ===============================
# 8. Train / Validation split
# ===============================
X_train, X_val, y_train, y_val = train_test_split(
    X, y_log, test_size=0.2, random_state=42
)

# X_train, X_val, y_train, y_val = train_test_split(
#     X, y, test_size=0.2, random_state=42
# )

# ===============================
# 9. CatBoost (best practice)
# ===============================
model = CatBoostRegressor(
    iterations=10000,
    learning_rate=0.02,
    depth=7,
    l2_leaf_reg=3,
    loss_function='RMSE',
    eval_metric='RMSE',

    task_type='GPU',          # 🔥 ВКЛЮЧАЕМ GPU
    devices='0',              # первый GPU
    gpu_ram_part=0.95,        # почти вся память

    random_seed=42,
    early_stopping_rounds=1000,
    verbose=500
)


model.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    cat_features=cat_features
)

# ===============================
# 10. Валидация
# ===============================
val_preds_log = model.predict(X_val)
val_preds = np.expm1(val_preds_log)

rmse_val = np.sqrt(mean_squared_error(np.expm1(y_val), val_preds))

# val_preds = model.predict(X_val)
# # val_preds = np.expm1(val_preds_log)

# rmse_val = np.sqrt(mean_squared_error(y_val, val_preds))

print(f'Validation RMSE: {rmse_val:.4f}')

# ===============================
# 11. Предсказание test
# ===============================
test_preds_log = model.predict(test)
test_preds = np.expm1(test_preds_log)

# ===============================
# 12. Submission
# ===============================
submission = pd.read_csv('solution_example_full.csv').copy()
submission[submission.columns[1]] = pd.Series(test_preds, index=submission.index, dtype='float64')
submission.to_csv('my_submission.csv', index=False)

print('Файл my_submission.csv сохранён')


C:\Users\yarik\AppData\Local\Temp\ipykernel_19260\4123585474.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=["object"]):
C:\Users\yarik\AppData\Local\Temp\ipykernel_19260\4123585474.py:26: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migr

После фильтрации выбросов осталось объектов: 100000
0:	learn: 0.3439235	test: 0.3413608	best: 0.3413608 (0)	total: 82.5ms	remaining: 13m 45s
500:	learn: 0.0139165	test: 0.0125056	best: 0.0125056 (500)	total: 30.7s	remaining: 9m 42s
1000:	learn: 0.0119188	test: 0.0115294	best: 0.0115294 (1000)	total: 1m	remaining: 9m 6s
1500:	learn: 0.0113348	test: 0.0113254	best: 0.0113254 (1500)	total: 1m 31s	remaining: 8m 38s
2000:	learn: 0.0110638	test: 0.0112334	best: 0.0112333 (1998)	total: 2m 5s	remaining: 8m 23s
2500:	learn: 0.0108414	test: 0.0111796	best: 0.0111796 (2500)	total: 2m 39s	remaining: 7m 56s
3000:	learn: 0.0106624	test: 0.0111470	best: 0.0111470 (2999)	total: 3m 16s	remaining: 7m 39s
3500:	learn: 0.0104992	test: 0.0111241	best: 0.0111241 (3500)	total: 3m 50s	remaining: 7m 7s
4000:	learn: 0.0103564	test: 0.0111071	best: 0.0111071 (4000)	total: 4m 23s	remaining: 6m 35s
4500:	learn: 0.0102197	test: 0.0110975	best: 0.0110975 (4500)	total: 4m 55s	remaining: 6m 1s
5000:	learn: 0.0100941	t

NameError: name 'sample_sub' is not defined

In [4]:
submission = pd.read_csv('solution_example_full.csv').copy()
submission[submission.columns[1]] = pd.Series(test_preds, index=submission.index, dtype='float64')
submission.to_csv('my_submission.csv', index=False)

print('Файл my_submission.csv сохранён')

TypeError: Invalid value '[18536581.68153245 12612549.48629696 22909556.63116412 ...
  9623069.30428145 24572416.89185756 26794266.74215376]' for dtype 'int64'